# Phase 9 – Customer Churn Prediction using XGBoost

The company wants to identify customers who are likely to stop purchasing so that retention campaigns can be launched before they are lost.

## Input

Customer-level feature dataset created during Feature Engineering.

## Output

- Churn Prediction
- Churn Probability
- Feature Importance

## Business Value

Allows the marketing team to proactively retain valuable customers and reduce revenue loss.

In [1]:
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier

import matplotlib.pyplot as plt

Mounted at /content/drive


In [2]:
model_data = pd.read_csv(
    "/content/drive/MyDrive/Revenue_Intelligence_Project/Data/model_ready_dataset.csv"
)

print(model_data.shape)

model_data.head()

(5881, 9)


,Recency,Frequency,Monetary,AverageOrderValue,CustomerLifetime,AverageDaysBetweenOrders,SegmentEncoded,CustomerAgeEncoded,Churn
0,158,8,2433.28,304.160000,581,82.714286,3,0,1
1,3,57,29018.06,509.088772,735,12.714286,1,0,0
2,449,2,613.08,306.540000,290,290.000000,2,1,1
3,1,145,580987.04,4006.807172,738,4.868056,1,0,0
4,4,52,22950.22,441.350385,735,13.901961,1,0,0


In [11]:
X = model_data[
    [
        "Frequency",
        "Monetary",
        "AverageOrderValue",
        "CustomerLifetime",
        "AverageDaysBetweenOrders",
        "CustomerAgeEncoded"
    ]
]

y = model_data["Churn"]

print(X.head())
print("\nShape:", X.shape)

   Frequency   Monetary  AverageOrderValue  CustomerLifetime  \
0          8    2433.28         304.160000               581   
1         57   29018.06         509.088772               735   
2          2     613.08         306.540000               290   
3        145  580987.04        4006.807172               738   
4         52   22950.22         441.350385               735   

   AverageDaysBetweenOrders  CustomerAgeEncoded  
0                 82.714286                   0  
1                 12.714286                   0  
2                290.000000                   1  
3                  4.868056                   0  
4                 13.901961                   0  

Shape: (5881, 6)


In [12]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y
)

In [13]:
print("Training Data:", X_train.shape)
print("Testing Data :", X_test.shape)

Training Data: (4704, 6)
Testing Data : (1177, 6)


In [14]:
# ==========================================
# Train XGBoost Model
# ==========================================

model = XGBClassifier(

    random_state=42,

    n_estimators=100,

    max_depth=4,

    learning_rate=0.1,

    objective="binary:logistic",

    eval_metric="logloss"
)

model.fit(X_train, y_train)

print("Model Training Completed!")

Model Training Completed!


In [15]:
# ==========================================
# Predict Test Dataset
# ==========================================

y_pred = model.predict(X_test)

y_prob = model.predict_proba(X_test)[:,1]

In [16]:
# ==========================================
# Model Evaluation
# ==========================================

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(y_test, y_pred)

recall = recall_score(y_test, y_pred)

f1 = f1_score(y_test, y_pred)

roc_auc = roc_auc_score(y_test, y_prob)

print(f"Accuracy : {accuracy:.4f}")

print(f"Precision: {precision:.4f}")

print(f"Recall   : {recall:.4f}")

print(f"F1 Score : {f1:.4f}")

print(f"ROC-AUC  : {roc_auc:.4f}")

Accuracy : 0.7349
Precision: 0.7145
Recall   : 0.7980
F1 Score : 0.7539
ROC-AUC  : 0.7938


In [17]:
cm = confusion_matrix(y_test, y_pred)

print(cm)

[[387 191]
 [121 478]]


In [18]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.76      0.67      0.71       578
           1       0.71      0.80      0.75       599

    accuracy                           0.73      1177
   macro avg       0.74      0.73      0.73      1177
weighted avg       0.74      0.73      0.73      1177



In [20]:
# ==========================================
# Save Trained Model
# ==========================================
import joblib

joblib.dump(
    model,
    "/content/drive/MyDrive/Revenue_Intelligence_Project/Data/xgboost_churn_model.pkl"
)

print("XGBoost Model Saved Successfully!")

XGBoost Model Saved Successfully!
